# Project Overview (PyBaMM + Neural ODE, 25°C)

This notebook is meant to make the repository **observable**: you can see what data exists, what preprocessing produces, and what the current model components do.

**Dataset constraint:** all tests are at **25°C** (treat as isothermal; no temperature generalisation claims).


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# --- Ensure we are at repo root ---
ROOT = Path.cwd()
if not (ROOT / "CLAUDE.md").exists() and (ROOT.parent / "CLAUDE.md").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

# Allow `import src.*` when running from notebooks/
sys.path.insert(0, str(ROOT))

# Matplotlib cache/config must be writable in many environments
os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Python:", sys.version.split()[0])


In [ ]:
# Versions of key libraries
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import yaml

try:
    import pybamm
    print("pybamm:", pybamm.__version__)
except Exception as e:
    print("pybamm import failed:", type(e).__name__, e)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)


## 1) What data is present? (manifest)

The bootstrap step generates a `data/processed/dataset_manifest.yaml` that records which test folders and cell files were discovered.


In [ ]:
manifest_path = Path("data/processed/dataset_manifest.yaml")
if not manifest_path.exists():
    print("Manifest missing. Run bootstrap first:")
    print("  .venv/bin/python src/bootstrap.py")
else:
    manifest = yaml.safe_load(manifest_path.read_text())
    tests = manifest.get("tests", {})
    rows = []
    for test_name, info in tests.items():
        rows.append(
            {
                "test": test_name,
                "path": info.get("path"),
                "n_cell_files": info.get("n_cell_files", 0),
                "cells": ",".join(info.get("cells", [])),
            }
        )
    df_manifest = pd.DataFrame(rows).sort_values(["test"]).reset_index(drop=True)
    display(df_manifest)


## 2) Cell selection (default cohort)

After importing additional cells, we generate an auditable cohort selection:

- Report: `data/processed/cell_selection_report.md`
- Config: `configs/dataset.yaml` (used by `src/bootstrap.py` as the default cell list)


In [ ]:
from pathlib import Path
import sys
import yaml
from IPython.display import Markdown, display

dataset_cfg = Path('configs/dataset.yaml')
selected_cells = None
splits = None
if dataset_cfg.exists():
    cfg = yaml.safe_load(dataset_cfg.read_text()) or {}
    selected_cells = cfg.get('dataset', {}).get('selected_cells')
    splits = cfg.get('splits', {})

print('dataset.yaml found =', dataset_cfg.exists())
print('selected_cells =', selected_cells)
print('splits =', splits)

report_path = Path('data/processed/cell_selection_report.md')
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print('Selection report missing. Generate it with:')
    print(f'  {sys.executable} src/cell_selection.py')


## 3) Experiment runs (local tracking)

Runs are stored under `outputs/experiments/<run_id>/` and include metadata, config snapshots, metrics JSONL, and artifacts.


In [ ]:
from src.experiment_tracking import ExperimentRun

exp_root = Path("outputs/experiments")
run_dirs = sorted([p for p in exp_root.glob("*") if p.is_dir()])
print("# runs:", len(run_dirs))
for p in run_dirs[-5:]:
    print("-", p.name)

if run_dirs:
    latest = run_dirs[-1]
    meta_path = latest / "meta.yaml"
    if meta_path.exists():
        meta = yaml.safe_load(meta_path.read_text())
        print("\nLatest run meta keys:", sorted(meta.keys()))
        print("Latest run name:", meta.get("name"))
        print("Created UTC:", meta.get("created_utc"))


## 4) (Optional) Re-run bootstrap

Bootstrap generates the first defensible, data-derived artifacts (OCV curves, SOH summaries, GITT step metrics) and plots into `outputs/results/`.


In [ ]:
import subprocess

# Bootstrap generates the first defensible artifacts + plots.
# It defaults to the selected cell cohort in configs/dataset.yaml (if present).
# Uncomment to re-run:
# subprocess.run([sys.executable, 'src/bootstrap.py'], check=True)
print('Bootstrap command:', f"{sys.executable} src/bootstrap.py")


## 5) OCV curves (processed)

From `data/processed/ocv_curves.parquet` (SOC is estimated if not provided in the raw export).


In [ ]:
ocv_path = Path('data/processed/ocv_curves.parquet')
ocv = pd.read_parquet(ocv_path)
if selected_cells:
    ocv = ocv[ocv['cell_id'].astype(str).isin([str(x) for x in selected_cells])]
display(ocv.head())
print('rows:', len(ocv), 'cells:', sorted(ocv['cell_id'].unique().tolist()))

fig, ax = plt.subplots(figsize=(6, 4))
for cid, g in ocv.groupby('cell_id'):
    ax.plot(g['soc'], g['voltage'], label=str(cid), linewidth=1.0)
ax.set(xlabel='SOC (estimated)', ylabel='Voltage [V]', title='OCV curves (25°C)')
ax.legend(title='cell_id', fontsize=8)
fig.tight_layout()
plt.show()


## 6) Capacity fade summaries (processed)

- `rpt_capacity_fade.parquet`: SOH normalised to the first RPT in each file
- `longterm_capacity_fade.parquet`: SOH normalised to the first long-term cycle in each file


In [ ]:
rpt = pd.read_parquet('data/processed/rpt_capacity_fade.parquet')
lt = pd.read_parquet('data/processed/longterm_capacity_fade.parquet')
if selected_cells:
    rpt = rpt[rpt['cell_id'].astype(str).isin([str(x) for x in selected_cells])]
    lt = lt[lt['cell_id'].astype(str).isin([str(x) for x in selected_cells])]

display(rpt)
display(lt.head())

fig, ax = plt.subplots(figsize=(6, 4))
for cid, g in lt.groupby('cell_id'):
    ax.plot(g['cycle_n'], g['SOH'], label=f"{cid} (longterm)")
ax.set(xlabel='Cycle number', ylabel='SOH', title='Long-term SOH (25°C)')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## 7) GITT step metrics (defensible outputs)

Computed by `src/param_id/gitt_ds.py` with the key constraint that **electrode-specific diffusivities are not claimed** from full-cell voltage alone.


In [ ]:
from pathlib import Path

def load_gitt_metrics(cell_id: str) -> pd.DataFrame:
    return pd.read_parquet(f"data/processed/gitt_metrics_cell_{cell_id}.parquet")

# Prefer the selected cohort from configs/dataset.yaml; otherwise discover from parquet files
cells = selected_cells
if not cells:
    cells = sorted([p.stem.split('_')[-1] for p in Path('data/processed').glob('gitt_metrics_cell_*.parquet')])

metrics = {cid: load_gitt_metrics(cid) for cid in cells}
for cid, df in metrics.items():
    print(cid, 'rows:', len(df), 'cols:', list(df.columns))

cid = cells[0]
df = metrics[cid]
display(df.head())

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df['cycle_step'], df['delta_Es_V'], label=r'$\Delta E_s$ [V]')
ax.plot(df['cycle_step'], df['delta_Etau_V'], label=r'$\Delta E_{\tau}$ [V]', alpha=0.8)
ax.set(xlabel='GITT step', ylabel='Voltage change [V]', title=f'GITT step metrics ({cid})')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## 8) Raw-vs-loaded schema peek (one file)

Shows the raw EVE export columns and what `src/data_loader.py` produces (column normalisation, time reconstruction, cell_id standardisation).


In [ ]:
raw = pd.read_csv("Data/GITT/EVE_GITT_cell_0005.csv")
print("raw columns:", raw.columns.tolist())
display(raw.head())

from src.data_loader import load_gitt_pulses
loaded = load_gitt_pulses("0005")
print("\nloaded columns:", loaded.columns.tolist())
display(loaded.head())
print("time monotonic:", bool((loaded["time"].diff().fillna(0) >= 0).all()))


## 9) Quick PyBaMM installation sanity check

This is **not** a calibrated cell model yet; it just confirms PyBaMM runs and can solve a small problem.


In [ ]:
import pybamm

model = pybamm.lithium_ion.DFN(options={"thermal": "isothermal"})
param = pybamm.ParameterValues("Prada2013")
sim = pybamm.Simulation(model, parameter_values=param)

# Solve a tiny time window (seconds) to validate the install.
sol = sim.solve([0, 10])
print("Solved.")
print("Terminal voltage @ t=0:", float(sol["Terminal voltage [V]"](0)))
print("Terminal voltage @ t=10s:", float(sol["Terminal voltage [V]"](10)))


## What is implemented vs. what is next

Implemented (observable now):
- Raw data discovery + standardisation (`src/data_loader.py`)
- Defensible GITT step metrics (`src/param_id/gitt_ds.py`)
- Local experiment tracking (`src/experiment_tracking.py`)
- Bootstrap artifacts + plots (`src/bootstrap.py`)

Next (to implement):
- Parameter identification beyond GITT step metrics (OCV fitting, HPPC/DCIR feature extraction)
- PyBaMM sweep dataset generation (`data/synthetic/trajectories.parquet`)
- PINN/Neural ODE training scripts (pretrain + fine-tune)
